In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

df = pd.read_csv("../data/cleaned.csv")
df['posting_date'] = pd.to_datetime(df['posting_date'], errors='coerce')
df['due_in_date'] = pd.to_datetime(df['due_in_date'], errors='coerce')
df['clear_date'] = pd.to_datetime(df['clear_date'], errors='coerce')

df['posting_month'] = df['posting_date'].dt.month.fillna(6)
df['delay_days'] = (df['clear_date'] - df['due_in_date']).dt.days
df['is_q_end'] = df['posting_month'].isin([3,6,9,12]).astype(int)
df['buisness_year'] = df['buisness_year'].fillna(2020)
df['total_open_amount'] = df['total_open_amount'].fillna(df['total_open_amount'].median())

# FIX: Target = clear_date khali hai toh risky (open invoice)
df['isOpen'] = df['clear_date'].isna().astype(int)
df['is_delayed'] = df['isOpen'] # ab ye balanced hoga

print(df['is_delayed'].value_counts())

features = ['total_open_amount','posting_month','delay_days','buisness_year','is_q_end']
# delay_days NaN ko 999 se fill (open invoices ka delay sabse zyada)
df['delay_days'] = df['delay_days'].fillna(999)
for c in features:
    df[c] = df[c].fillna(df[c].median())

X = df[features]
y = df['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=150, random_state=42)
model.fit(X_train, y_train)

print(f"Accuracy: {model.score(X_test, y_test):.3f}")
print(f"Classes: {model.classes_}")

joblib.dump(model, "../models/risk_model.pkl")
print("SAVED models/risk_model.pkl")

is_delayed
1    33571
0    15268
Name: count, dtype: int64
Accuracy: 1.000
Classes: [0 1]
SAVED models/risk_model.pkl


In [6]:
import pickle
with open("../models/model.pkl", "wb") as f:
    pickle.dump(model, f, protocol=4)

with open("../models/risk_model.pkl", "wb") as f:
    pickle.dump(model, f, protocol=4)